# Advanced ML Concepts - Solutions

Complete implementations of:
- Grouped Query Attention (GQA)
- Mixture of Experts (MoE)
- Manual KV Caching
- Encoder-Decoder & Cross-Attention
- Data Preprocessing & PyTorch DataLoaders
- LoRA (Low-Rank Adaptation)
- Multi-Label Classification

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss
import math

## Solution 1: Grouped Query Attention (GQA)

GQA reduces memory and computation by having multiple query heads share the same key and value heads.
If we have 8 query heads and 2 KV heads, each KV head is shared by 4 query heads.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads):
        super().__init__()
        assert d_model % num_query_heads == 0, "d_model must be divisible by num_query_heads"
        assert num_query_heads % num_kv_heads == 0, "num_query_heads must be divisible by num_kv_heads"
        
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.num_queries_per_kv = num_query_heads // num_kv_heads
        
        # Projections
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(d_model, d_model)
        
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        q = self.q_proj(x)  # (batch, seq_len, d_model)
        k = self.k_proj(x)  # (batch, seq_len, num_kv_heads * head_dim)
        v = self.v_proj(x)  # (batch, seq_len, num_kv_heads * head_dim)
        
        # Reshape Q for multi-head
        q = q.view(batch_size, seq_len, self.num_query_heads, self.head_dim)
        q = q.transpose(1, 2)  # (batch, num_query_heads, seq_len, head_dim)
        
        # Reshape K, V for KV heads
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        k = k.transpose(1, 2)  # (batch, num_kv_heads, seq_len, head_dim)
        
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        v = v.transpose(1, 2)  # (batch, num_kv_heads, seq_len, head_dim)
        
        # Repeat K and V to match the number of query heads
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)  # (batch, num_query_heads, seq_len, head_dim)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)  # (batch, num_query_heads, seq_len, head_dim)
        
        # Compute attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax and weighted sum
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)  # (batch, num_query_heads, seq_len, head_dim)
        
        # Concatenate heads
        output = output.transpose(1, 2).contiguous()  # (batch, seq_len, num_query_heads, head_dim)
        output = output.view(batch_size, seq_len, self.d_model)
        
        # Output projection
        output = self.out_proj(output)
        return output

# Test GQA
d_model = 512
num_query_heads = 8
num_kv_heads = 2
batch_size = 2
seq_len = 10

gqa = GroupedQueryAttention(d_model, num_query_heads, num_kv_heads)
x = torch.randn(batch_size, seq_len, d_model)
output = gqa(x)
print(f"GQA output shape: {output.shape}")
print(f"Expected shape: (batch={batch_size}, seq_len={seq_len}, d_model={d_model})")

## Solution 2: Mixture of Experts (MoE)

MoE routes each input to a subset of expert networks based on a learned gating function.

In [ ]:
class Expert(nn.Module):
    """Individual expert network (2-layer MLP)"""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.activation = nn.ReLU()
    
    def forward(self, x):
        x = self.activation(self.fc1(x))
        return self.fc2(x)

class MixtureOfExperts(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # Gating network
        self.gate = nn.Linear(input_dim, num_experts)
        
        # Expert networks
        self.experts = nn.ModuleList([
            Expert(input_dim, hidden_dim, output_dim) for _ in range(num_experts)
        ])
    
    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        batch_size, seq_len, input_dim = x.shape
        
        # Compute gate scores for each expert
        gate_logits = self.gate(x)  # (batch, seq_len, num_experts)
        
        # Select top-k experts
        top_k_logits, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        
        # Normalize weights for selected experts
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # (batch, seq_len, top_k)
        
        # Initialize output
        output = torch.zeros(batch_size, seq_len, self.experts[0].fc2.out_features, device=x.device)
        
        # Compute weighted sum of expert outputs
        for i in range(self.top_k):
            expert_idx = top_k_indices[:, :, i]  # (batch, seq_len)
            expert_weight = top_k_weights[:, :, i]  # (batch, seq_len)
            
            # Process each expert
            for expert_id in range(self.num_experts):
                # Create mask for tokens that use this expert
                mask = (expert_idx == expert_id)
                if mask.any():
                    expert_out = self.experts[expert_id](x)
                    # Apply weighted contribution
                    output += expert_out * expert_weight.unsqueeze(-1) * mask.unsqueeze(-1).float()
        
        return output

# Test MoE
input_dim = 128
hidden_dim = 256
output_dim = 128
num_experts = 4
top_k = 2

moe = MixtureOfExperts(input_dim, hidden_dim, output_dim, num_experts, top_k)
x = torch.randn(2, 10, input_dim)
output = moe(x)
print(f"MoE output shape: {output.shape}")
print(f"Using {top_k} out of {num_experts} experts per token")

## Solution 3: Manual KV Caching

KV caching stores key and value tensors from previous tokens to avoid recomputation during autoregressive generation.

In [ ]:
class KVCache:
    """Simple key-value cache for multiple layers"""
    def __init__(self):
        self.cache = {}  # layer_idx -> (k, v)
    
    def update(self, k, v, layer_idx):
        """Update cache for a specific layer"""
        if layer_idx in self.cache:
            # Concatenate with existing cache
            cached_k, cached_v = self.cache[layer_idx]
            k = torch.cat([cached_k, k], dim=2)  # Concatenate along sequence dimension
            v = torch.cat([cached_v, v], dim=2)
        self.cache[layer_idx] = (k, v)
        return k, v
    
    def get(self, layer_idx):
        """Retrieve cached K, V for a layer"""
        return self.cache.get(layer_idx, (None, None))
    
    def clear(self):
        """Clear all cached values"""
        self.cache = {}

class AttentionWithCache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x, cache=None, layer_idx=0, use_cache=False):
        batch_size, seq_len, _ = x.shape
        
        # Compute Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Reshape for multi-head attention
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Use cache if provided
        if cache is not None and use_cache:
            k, v = cache.update(k, v, layer_idx)
        
        # Compute attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        
        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.out_proj(output)
        
        return output

# Test KV caching
d_model = 256
num_heads = 4
attn = AttentionWithCache(d_model, num_heads)
cache = KVCache()

# First token
x1 = torch.randn(1, 1, d_model)
out1 = attn(x1, cache=cache, layer_idx=0, use_cache=True)
print(f"First token output shape: {out1.shape}")

# Second token (uses cached K, V from first token)
x2 = torch.randn(1, 1, d_model)
out2 = attn(x2, cache=cache, layer_idx=0, use_cache=True)
print(f"Second token output shape: {out2.shape}")
print(f"Cached K shape at layer 0: {cache.get(0)[0].shape}")

## Solution 4: Encoder-Decoder with Cross-Attention

Cross-attention allows the decoder to attend to encoder outputs, essential for sequence-to-sequence tasks.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # Project
        q = self.q_proj(q).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        
        # Concatenate and project
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.out_proj(output)

class CrossAttention(nn.Module):
    """Cross-attention: Q from decoder, K and V from encoder"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
    
    def forward(self, decoder_hidden, encoder_output, mask=None):
        # Q from decoder, K and V from encoder
        return self.attn(decoder_hidden, encoder_output, encoder_output, mask)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = CrossAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Self-attention with residual
        self_attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))
        
        # Cross-attention with residual
        cross_attn_output = self.cross_attn(x, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))
        
        # Feed-forward with residual
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        
        return x

# Test Encoder-Decoder
d_model = 256
num_heads = 8
d_ff = 1024
batch_size = 2
src_seq_len = 10
tgt_seq_len = 8

encoder_layer = EncoderLayer(d_model, num_heads, d_ff)
decoder_layer = DecoderLayer(d_model, num_heads, d_ff)

src = torch.randn(batch_size, src_seq_len, d_model)
tgt = torch.randn(batch_size, tgt_seq_len, d_model)

encoder_output = encoder_layer(src)
decoder_output = decoder_layer(tgt, encoder_output)

print(f"Encoder output shape: {encoder_output.shape}")
print(f"Decoder output shape: {decoder_output.shape}")

## Solution 5: Data Preprocessing & PyTorch DataLoaders

Proper data preprocessing and efficient data loading are crucial for training ML models.

In [ ]:
# Generate sample data
np.random.seed(42)
data = {
    'feature1': np.random.randn(1000),
    'feature2': np.random.randn(1000) * 10 + 5,
    'feature3': np.random.randn(1000) * 0.5,
    'feature4': np.random.choice([np.nan, 1, 2, 3], 1000),
    'label': np.random.randint(0, 3, 1000)
}
df = pd.DataFrame(data)

print(f"Original data shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

# Drop rows with missing values
df_clean = df.dropna()
print(f"\nCleaned data shape: {df_clean.shape}")

# Separate features and labels
X = df_clean.drop('label', axis=1).values
y = df_clean['label'].values

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply StandardScaler to features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")
print(f"\nFeature means after scaling (should be ~0): {X_train_scaled.mean(axis=0)}")
print(f"Feature stds after scaling (should be ~1): {X_train_scaled.std(axis=0)}")

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create datasets
train_dataset = CustomDataset(X_train_scaled, y_train)
test_dataset = CustomDataset(X_test_scaled, y_test)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Test DataLoader
for batch_features, batch_labels in train_loader:
    print(f"Batch features shape: {batch_features.shape}")
    print(f"Batch labels shape: {batch_labels.shape}")
    print(f"Sample features: {batch_features[0]}")
    print(f"Sample label: {batch_labels[0]}")
    break

## Solution 6: LoRA (Low-Rank Adaptation)

LoRA adds trainable low-rank matrices to frozen pretrained weights, enabling efficient fine-tuning.

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # Frozen base layer
        self.base_layer = nn.Linear(in_features, out_features)
        self.base_layer.weight.requires_grad = False
        self.base_layer.bias.requires_grad = False
        
        # Low-rank adaptation matrices
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
    
    def forward(self, x):
        # Base output (frozen)
        base_out = self.base_layer(x)
        
        # LoRA adaptation: x @ A @ B
        lora_out = (x @ self.lora_A @ self.lora_B) * self.scaling
        
        return base_out + lora_out

# Create example model with LoRA
in_features = 512
out_features = 512
rank = 8

lora_layer = LoRALayer(in_features, out_features, rank=rank)

# Count parameters
total_params = sum(p.numel() for p in lora_layer.parameters())
trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (LoRA): {trainable_params:,}")
print(f"Frozen parameters (base): {frozen_params:,}")
print(f"\nParameter reduction: {100 * trainable_params / total_params:.2f}% trainable")

# Verify only LoRA parameters require gradients
print("\nTrainable parameters:")
for name, param in lora_layer.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

# Test forward pass
x = torch.randn(2, 10, in_features)
output = lora_layer(x)
print(f"\nOutput shape: {output.shape}")

## Solution 7: Multi-Label Classifier

Multi-label classification allows each sample to have multiple labels (not mutually exclusive).

In [ ]:
# Generate multi-label dataset
n_samples = 1000
n_features = 20
n_labels = 5

np.random.seed(42)
X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 2, (n_samples, n_labels))  # Binary labels for each class

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train shape: {X_train.shape}, {y_train.shape}")
print(f"Test shape: {X_test.shape}, {y_test.shape}")
print(f"\nSample labels (multiple per instance): {y_train[0]}")

In [ ]:
class MultiLabelClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_labels):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, num_labels)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)  # No activation - BCEWithLogitsLoss applies sigmoid
        return x

# Create model
model = MultiLabelClassifier(input_dim=n_features, hidden_dim=64, num_labels=n_labels)

# BCEWithLogitsLoss combines sigmoid and BCE for numerical stability
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Model architecture:")
print(model)

In [ ]:
# Create datasets and dataloaders
class MultiLabelDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels)  # Float for BCE loss
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = MultiLabelDataset(X_train, y_train)
test_dataset = MultiLabelDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Training loop
num_epochs = 20
train_losses = []
test_losses = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for features, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            # Apply sigmoid and threshold at 0.5
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.append(preds)
            all_labels.append(labels)
    
    test_loss /= len(test_loader)
    test_losses.append(test_loss)
    
    # Calculate accuracy metrics
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    # Exact match ratio (all labels correct)
    exact_match = (all_preds == all_labels).all(axis=1).mean()
    # Hamming loss (fraction of wrong labels)
    ham_loss = hamming_loss(all_labels, all_preds)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
        print(f"  Exact Match: {exact_match:.4f}, Hamming Loss: {ham_loss:.4f}")

print("\nTraining complete!")
print(f"Final Exact Match Accuracy: {exact_match:.4f}")
print(f"Final Hamming Loss: {ham_loss:.4f}")

## Bonus: Integrate Multiple Concepts

Combining GQA with LoRA for efficient attention fine-tuning.

In [ ]:
class GQAWithLoRA(nn.Module):
    """Grouped Query Attention with LoRA adaptation on projections"""
    def __init__(self, d_model, num_query_heads, num_kv_heads, lora_rank=8):
        super().__init__()
        assert d_model % num_query_heads == 0
        assert num_query_heads % num_kv_heads == 0
        
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.num_queries_per_kv = num_query_heads // num_kv_heads
        
        # Apply LoRA to query, key, value projections
        self.q_proj = LoRALayer(d_model, d_model, rank=lora_rank)
        self.k_proj = LoRALayer(d_model, num_kv_heads * self.head_dim, rank=lora_rank)
        self.v_proj = LoRALayer(d_model, num_kv_heads * self.head_dim, rank=lora_rank)
        self.out_proj = LoRALayer(d_model, d_model, rank=lora_rank)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project using LoRA-enhanced layers
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Reshape for multi-head attention
        q = q.view(batch_size, seq_len, self.num_query_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # Repeat K and V for grouped queries
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        
        # Compute attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        
        # Concatenate and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.out_proj(output)
        
        return output

# Test GQA with LoRA
d_model = 512
num_query_heads = 8
num_kv_heads = 2
lora_rank = 16

gqa_lora = GQAWithLoRA(d_model, num_query_heads, num_kv_heads, lora_rank)

# Count parameters
total_params = sum(p.numel() for p in gqa_lora.parameters())
trainable_params = sum(p.numel() for p in gqa_lora.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (LoRA): {trainable_params:,}")
print(f"Parameter reduction: {100 * trainable_params / total_params:.2f}% trainable")

# Test forward pass
x = torch.randn(2, 10, d_model)
output = gqa_lora(x)
print(f"\nOutput shape: {output.shape}")
print("\nSuccessfully combined GQA with LoRA for efficient fine-tuning!")